In [ ]:
import os
import numpy as np
import SimpleITK as sitk
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
from scipy.ndimage import label, generate_binary_structure
import logging

sitk.ProcessObject.SetGlobalWarningDisplay(False)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def calculate_suv_peak(image_arr, mask_arr, spacing, radius_mm=6.2035):
    """Calcule le SUV Peak (sphère de ~1 mL autour du voxel le plus chaud du masque)."""
    masked_img = np.where(mask_arr > 0, image_arr, -np.inf)
    if np.all(masked_img == -np.inf): 
        return np.nan
        
    idx_z, idx_y, idx_x = np.unravel_index(np.argmax(masked_img), masked_img.shape)
    sp_x, sp_y, sp_z = spacing
    
    rad_z = int(np.ceil(radius_mm / sp_z))
    rad_y = int(np.ceil(radius_mm / sp_y))
    rad_x = int(np.ceil(radius_mm / sp_x))
    
    z_min = max(0, idx_z - rad_z)
    z_max = min(image_arr.shape[0], idx_z + rad_z + 1)
    y_min = max(0, idx_y - rad_y)
    y_max = min(image_arr.shape[1], idx_y + rad_y + 1)
    x_min = max(0, idx_x - rad_x)
    x_max = min(image_arr.shape[2], idx_x + rad_x + 1)
    
    box_arr = image_arr[z_min:z_max, y_min:y_max, x_min:x_max]
    zz, yy, xx = np.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
    
    dist2 = ((zz - idx_z) * sp_z)**2 + ((yy - idx_y) * sp_y)**2 + ((xx - idx_x) * sp_x)**2
    sphere_mask = dist2 <= (radius_mm**2)
    
    if not np.any(sphere_mask):
        return np.nan
    
    return np.mean(box_arr[sphere_mask])


def calculate_metrics(gt_arr, pred_arr, mask_arr, spacing):
    voxels_gt = gt_arr[mask_arr > 0]
    voxels_pred = pred_arr[mask_arr > 0]
    
    if len(voxels_gt) == 0:
        return None
        
    # --- aRE ---
    voxels_gt_safe = np.where(voxels_gt == 0, 1e-8, voxels_gt)
    are_val = np.mean(np.abs((voxels_pred - voxels_gt_safe) / voxels_gt_safe)) * 100
    
    # --- PSNR ---
    mse = np.mean((voxels_gt - voxels_pred) ** 2)
    max_val = np.max(voxels_gt)
    psnr_val = 10 * np.log10((max_val ** 2) / mse) if mse > 1e-8 else np.inf
    
    # --- COV (Coefficient of Variation) ---
    mean_gt, mean_pred = np.mean(voxels_gt), np.mean(voxels_pred)
    cov_gt = (np.std(voxels_gt) / mean_gt) if mean_gt > 1e-8 else 0
    cov_pred = (np.std(voxels_pred) / mean_pred) if mean_pred > 1e-8 else 0
    
    # --- Max ---
    max_gt, max_pred = np.max(voxels_gt), np.max(voxels_pred)
    
    # --- SUV Peak ---
    suv_peak_gt = calculate_suv_peak(gt_arr, mask_arr, spacing)
    suv_peak_pred = calculate_suv_peak(pred_arr, mask_arr, spacing)
    
    return {
        "aRE": are_val,
        "PSNR": psnr_val,
        "COV_GT": cov_gt,
        "COV_Pred": cov_pred,
        "Mean_GT": mean_gt,
        "Mean_Pred": mean_pred,
        "Max_GT": max_gt,
        "Max_Pred": max_pred,
        "SUV_Peak_GT": suv_peak_gt,
        "SUV_Peak_Pred": suv_peak_pred
    }


def process_single_subject(args):
    domain, subject_id, subject_path, gt_filenames, pseudo_filenames, gauss_filenames, std_filename, mask_filename, target_vois = args
    
    subject_results = {}
    
    # Liste des dossiers de VOIs physiquement présents pour ce patient
    available_vois = [f for f in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, f))]
    
    # Nettoyage au cas où les VOIs en entrée contiendraient ".nii.gz"
    target_vois_clean = [v.replace('.nii.gz', '').replace('.nii', '') for v in target_vois]
    
    for std_key, gt_file in gt_filenames.items():
        std_results = {}
        
        methods_to_eval = {
            'PseudoEARL-Net': pseudo_filenames.get(std_key),
            'Gaussian-EARL': gauss_filenames.get(std_key),
            'Standard': std_filename
        }
        
        for voi_target in target_vois_clean:
            voi_folder = next((v for v in available_vois if v.lower() == voi_target.lower()), None)
            if not voi_folder:
                continue
                
            voi_path = os.path.join(subject_path, voi_folder)
            
            gt_path = os.path.join(voi_path, gt_file)
            mask_path = os.path.join(voi_path, mask_filename)
            
            if not os.path.exists(gt_path) or not os.path.exists(mask_path):
                continue
                
            try:
                gt_img = sitk.ReadImage(gt_path)
                gt_arr = sitk.GetArrayFromImage(gt_img)
                mask_img = sitk.ReadImage(mask_path)
                mask_arr = sitk.GetArrayFromImage(mask_img)
                spacing = gt_img.GetSpacing()
                
                is_lesion = voi_target.lower() in ['lesion', 'lesions']
                voi_metrics = {}
                
                # ==========================================
                # TRAITEMENT LÉSION : Multi-Composantes
                # ==========================================
                if is_lesion:
                    struct_3d = generate_binary_structure(3, 3)
                    labeled_arr, num_features = label(mask_arr > 0, structure=struct_3d)
                    
                    unique_labels, counts = np.unique(labeled_arr, return_counts=True)
                    size_dict = dict(zip(unique_labels, counts))
                    
                    for comp_idx in range(1, num_features + 1):
                        # Filtrage du bruit (< 10 voxels)
                        if size_dict.get(comp_idx, 0) < 10:
                            continue
                            
                        comp_mask = (labeled_arr == comp_idx).astype(np.uint8)
                        comp_metrics = {}
                        
                        for method_name, pred_file in methods_to_eval.items():
                            if not pred_file: continue
                            pred_path = os.path.join(voi_path, pred_file)
                            if not os.path.exists(pred_path): continue
                            
                            pred_arr = sitk.GetArrayFromImage(sitk.ReadImage(pred_path))
                            metrics = calculate_metrics(gt_arr, pred_arr, comp_mask, spacing)
                            
                            if metrics:
                                comp_metrics[method_name] = metrics
                                
                        if comp_metrics:
                            voi_metrics[comp_idx] = comp_metrics
                            
                # ==========================================
                # TRAITEMENT CLASSIQUE : Masque Global
                # ==========================================
                else:
                    for method_name, pred_file in methods_to_eval.items():
                        if not pred_file: continue
                        pred_path = os.path.join(voi_path, pred_file)
                        if not os.path.exists(pred_path): continue
                        
                        pred_arr = sitk.GetArrayFromImage(sitk.ReadImage(pred_path))
                        metrics = calculate_metrics(gt_arr, pred_arr, mask_arr, spacing)
                        
                        if metrics:
                            voi_metrics[method_name] = metrics
                            
                if voi_metrics:
                    std_results[voi_target] = voi_metrics
                    
            except Exception as e:
                pass # Silencieux pour ne pas crasher le multiprocessing
        
        if std_results:
            subject_results[std_key] = std_results
            
    return domain, subject_id, subject_results


# ==============================================================================
# FONCTION PRINCIPALE D'EXTRACTION
# ==============================================================================
def extract_comprehensive_metrics(
    base_dirs, # ex: ['data/PET-EARL/domain_a100', 'data/PET-EARL/domain_chb']
    gt_filenames={'earl1': 'earl1.nii.gz', 'earl2': 'earl2.nii.gz'},
    pseudo_filenames={'earl1': 'pseudo-earl1.nii.gz', 'earl2': 'pseudo-earl2.nii.gz'},
    gauss_filenames={'earl1': 'gaussian-earl1.nii.gz', 'earl2': 'gaussian-earl2.nii.gz'},
    std_filename='pet.nii.gz',
    mask_filename='mask.nii.gz', 
    target_vois=['liver', 'lung', 'brain', 'spleen', 'urinary_bladder', 'lesion'],
    num_workers=32
):
    if isinstance(base_dirs, str):
        base_dirs = [base_dirs]
        
    tasks = []
    for domain_dir in base_dirs:
        if not os.path.exists(domain_dir):
            logging.warning(f"Le dossier domaine {domain_dir} n'existe pas.")
            continue
            
        domain_name = os.path.basename(domain_dir)
        subjects = [s for s in os.listdir(domain_dir) if os.path.isdir(os.path.join(domain_dir, s))]
        
        for subj in subjects:
            subject_path = os.path.join(domain_dir, subj)
            tasks.append((
                domain_name, subj, subject_path,
                gt_filenames, pseudo_filenames, gauss_filenames,
                std_filename, mask_filename, target_vois
            ))
            
    master_dict = {}
    print(f"\n🚀 Lancement de l'extraction sur {len(tasks)} patients.")
    
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_subject, task): task for task in tasks}
        
        for future in tqdm(as_completed(futures), total=len(tasks), desc="Extraction des Métriques"):
            domain, subject_id, subj_results = future.result()
            
            if subj_results:
                if domain not in master_dict:
                    master_dict[domain] = {}
                master_dict[domain][subject_id] = subj_results

    print("✅ Extraction terminée.")
    return master_dict


final_dict = extract_comprehensive_metrics(
    base_dirs=[
        './outputs/pseudo-earl/a100',
        './outputs/pseudo-earl/chb',
        './outputs/pseudo-earl/rennes',
        './outputs/pseudo-earl/nantes'
    ],
    mask_filename='mask.nii.gz', # Modifie si ton masque s'appelle différemment
    target_vois=['liver', 'lung', 'brain', 'spleen', 'urinary_bladder', 'lesion']
)

# # save as pickle
import pickle
with open('extracted_metrics.pkl', 'wb') as f:
    pickle.dump(final_dict, f)


🚀 Lancement de l'extraction sur 1236 patients.


Extraction des Métriques: 100%|██████████| 1236/1236 [02:47<00:00,  7.36it/s] 

✅ Extraction terminée.


In [17]:
## load
import pickle
with open('extracted_metrics.pkl', 'rb') as f:
    final_dict = pickle.load(f)

In [18]:
import pandas as pd
import numpy as np

def create_summary_table(final_dict, target_method="PseudoEARL-Net"):
    """
    Extrait les aRE d'un dictionnaire, gère les sous-composantes des lésions,
    agrège par (Centre, norme EARL), et génère un DataFrame 'moyenne ± std'.
    """
    
    records = []
    
    # 1. Parcours du dictionnaire et extraction
    for center, subjects in final_dict.items():
        for subj_id, earls in subjects.items():
            for earl_ver, vois in earls.items():
                row = {
                    'Center_raw': center,
                    'Subject': subj_id,
                    'EARL_raw': earl_ver
                }
                
                for voi, content in vois.items():
                    # --- CAS PARTICULIER : LESIONS (Multi-composantes) ---
                    if voi == 'lesion':
                        patient_lesion_ares = []
                        # content ressemble à {1: {'PseudoEARL-Net': {'aRE': 1.26...}}, 2: {...}}
                        for comp_id, methods in content.items():
                            if target_method in methods and 'aRE' in methods[target_method]:
                                patient_lesion_ares.append(methods[target_method]['aRE'])
                        
                        # On moyenne toutes les lésions du patient pour avoir 1 seule valeur représentative
                        if patient_lesion_ares:
                            row[voi] = np.mean(patient_lesion_ares)
                            
                    # --- CAS STANDARD : ORGANES (liver, brain, lung...) ---
                    else:
                        # content ressemble à {'PseudoEARL-Net': {'aRE': 0.80...}, 'Gaussian-EARL': {...}}
                        if target_method in content and 'aRE' in content[target_method]:
                            row[voi] = content[target_method]['aRE']
                            
                records.append(row)
                
    df_flat = pd.DataFrame(records)
    
    # 2. Dictionnaires de mapping (Traduction vers LaTeX)
    center_map = {
        'chb': 'Rouen$_1$', 
        'a100': 'Rouen$_2$', 
        'rennes': 'Rennes', 
        'nantes': 'Nantes'
    }
    
    earl_map = {
        'earl1': '1', 
        'earl2': '2'
    }
    
    voi_map = {
        'brain': 'Brain', 
        'liver': 'Liver', 
        'lung': 'Lung', 
        'spleen': 'Spleen', 
        'urinary_bladder': 'Bladder', 
        'lesion': 'Lesions'
    }

    # Application du mapping
    df_flat['Center'] = df_flat['Center_raw'].map(center_map)
    df_flat['EARL'] = df_flat['EARL_raw'].map(earl_map)
    df_flat = df_flat.rename(columns=voi_map)
    
    # 3. Calcul des Moyennes et Écarts-types (Mean / Std)
    groupby_cols = ['EARL', 'Center']
    target_vois = list(voi_map.values())
    
    # Calcul des stats sur les patients
    df_grouped = df_flat.groupby(groupby_cols)[target_vois].agg(['mean', 'std']).reset_index()
    
    # 4. Formatage en chaînes de caractères "Mean ± Std"
    df_final = pd.DataFrame()
    df_final['EARL'] = df_grouped['EARL']
    df_final['Center'] = df_grouped['Center']
    
    for voi in target_vois:
        mean_s = df_grouped[(voi, 'mean')]
        std_s = df_grouped[(voi, 'std')]
        
        formatted_col = []
        for m, s in zip(mean_s, std_s):
            # Gestion des cas où un masque n'existe pas du tout pour un centre
            if pd.isna(m) or pd.isna(s):
                formatted_col.append("N/A")
            else:
                formatted_col.append(f"{m:.2f} ± {s:.2f}")
        
        df_final[voi] = formatted_col

    # 5. Réorganisation des lignes pour correspondre exactement à ton LaTeX
    order_keys = [
        ('1', 'Rouen$_1$'), 
        ('1', 'Rouen$_2$'), 
        ('2', 'Rouen$_2$'), 
        ('1', 'Rennes'), 
        ('2', 'Nantes')
    ]
    
    df_final['_sort_key'] = df_final.apply(lambda row: (row['EARL'], row['Center']), axis=1)
    df_final['_sort_idx'] = df_final['_sort_key'].map({k: i for i, k in enumerate(order_keys)})
    
    # Tri et nettoyage
    df_final = df_final.sort_values('_sort_idx').drop(columns=['_sort_key', '_sort_idx']).reset_index(drop=True)
    
    return df_final


# Exécution
df_table = create_summary_table(final_dict, target_method="PseudoEARL-Net")
# print(df_table.to_string(index=False))
df_table

,EARL,Center,Brain,Liver,Lung,Spleen,Bladder,Lesions
0,1,Rouen$_1$,0.53 ± 0.76,0.28 ± 0.03,0.41 ± 0.05,0.28 ± 0.06,0.55 ± 0.36,0.32 ± 0.19
1,1,Rouen$_2$,0.55 ± 0.06,0.51 ± 0.06,0.89 ± 0.19,0.57 ± 0.10,1.15 ± 0.64,1.05 ± 0.43
2,2,Rouen$_2$,0.96 ± 0.20,1.12 ± 0.23,1.82 ± 0.36,1.31 ± 0.30,1.63 ± 0.71,1.81 ± 0.60
3,1,Rennes,0.94 ± 0.17,1.59 ± 0.25,2.18 ± 0.40,1.88 ± 0.36,2.07 ± 0.81,2.64 ± 1.23
4,2,Nantes,0.27 ± 0.04,0.26 ± 0.03,0.43 ± 0.13,0.27 ± 0.04,0.45 ± 0.16,0.51 ± 0.34


In [19]:
import pandas as pd
import numpy as np

def generate_are_psnr_table(master_dict):
    records = []
    
    # Définition stricte des centres par Standard
    valid_domains = {
        'earl1': ['rennes', 'chb', 'a100', 'domain_rennes', 'domain_chb', 'domain_a100'],
        'earl2': ['nantes', 'a100', 'domain_nantes', 'domain_a100']
    }
    
    # 1. Aplatissement du dictionnaire
    for domain, subjects in master_dict.items():
        domain_clean = domain.lower()
        
        for subj, std_data in subjects.items():
            for std_key, voi_data in std_data.items(): # std_key = 'earl1' ou 'earl2'
                
                # Filtrage strict du centre selon le standard cible
                if domain_clean not in valid_domains.get(std_key, []):
                    continue
                    
                target_label = std_key.upper()
                
                for voi, methods_data in voi_data.items():
                    # --- Gestion des Lésions (Multi-Composantes) ---
                    if voi.lower() in ['lesion', 'lesions']:
                        display_voi = 'Lesions'
                        # methods_data est de type {comp_idx: {method: metrics}}
                        for comp_idx, comp_methods in methods_data.items():
                            for method, metrics in comp_methods.items():
                                if method == 'Standard': continue
                                
                                # Nom de méthode formatté comme sur l'image
                                display_method = f"Gaussian-{target_label}" if "Gaussian" in method else "PseudoEARL-Net"
                                
                                records.append({
                                    'Target': target_label,
                                    'Method': display_method,
                                    'VOI': display_voi,
                                    'aRE': metrics['aRE'],
                                    'PSNR': metrics['PSNR']
                                })
                                
                    # --- Gestion des autres Organes (Global) ---
                    else:
                        display_voi = 'Bladder' if 'bladder' in voi.lower() else voi.capitalize()
                        
                        for method, metrics in methods_data.items():
                            if method == 'Standard': continue
                            
                            display_method = f"Gaussian-{target_label}" if "Gaussian" in method else "PseudoEARL-Net"
                            
                            records.append({
                                'Target': target_label,
                                'Method': display_method,
                                'VOI': display_voi,
                                'aRE': metrics['aRE'],
                                'PSNR': metrics['PSNR']
                            })
                            
    # 2. Création du DataFrame et Moyenne
    df_flat = pd.DataFrame(records)
    
    if df_flat.empty:
        print("⚠️ Aucune donnée ne correspond aux critères de filtrage.")
        return None
        
    df_agg = df_flat.groupby(['Target', 'Method', 'VOI']).mean().reset_index()
    
    # 3. Pivot pour correspondre à l'image (Tableau croisé)
    table = df_agg.pivot(index=['Target', 'Method'], columns='VOI', values=['aRE', 'PSNR'])
    
    # 4. Esthétique : Réorganisation des colonnes et des index
    table = table.swaplevel(axis=1) # Inverse l'ordre (VOI au dessus de la métrique)
    
    # Ordre spécifique des VOIs et métriques
    ordered_vois = ['Brain', 'Liver', 'Lung', 'Spleen', 'Bladder', 'Lesions']
    ordered_metrics = ['aRE', 'PSNR']
    
    # Filtre les VOIs qui sont réellement présentes dans les données pour éviter les KeyError
    existing_vois = [v for v in ordered_vois if v in table.columns.levels[0]]
    
    table = table.reindex(columns=pd.MultiIndex.from_product([existing_vois, ordered_metrics]))
    
    # Ordre spécifique des lignes
    ordered_methods = [
        ('EARL1', 'Gaussian-EARL1'), ('EARL1', 'PseudoEARL-Net'),
        ('EARL2', 'Gaussian-EARL2'), ('EARL2', 'PseudoEARL-Net')
    ]
    existing_methods = [m for m in ordered_methods if m in table.index]
    table = table.reindex(existing_methods)
    
    return table

# =========================================================
# EXÉCUTION
# =========================================================
# (On suppose que final_dict est le résultat retourné par ta fonction d'extraction)

summary_table = generate_are_psnr_table(final_dict)
summary_table

Brain                Liver                 Lung  \
                            aRE       PSNR       aRE       PSNR       aRE   
Target Method                                                               
EARL1  Gaussian-EARL1  2.401495  37.657435  2.405580  38.528708  3.335654   
       PseudoEARL-Net  0.662220  47.600257  0.735978  49.723702  1.122721   
EARL2  Gaussian-EARL2  1.190543  53.201315  1.151450  55.699148  1.752524   
       PseudoEARL-Net  0.502331  51.046346  0.545870  50.372343  0.892658   

                                    Spleen              Bladder             \
                            PSNR       aRE       PSNR       aRE       PSNR   
Target Method                                                                
EARL1  Gaussian-EARL1  41.669239  2.687032  36.647185  6.363896  36.661218   
       PseudoEARL-Net  52.415595  0.839993  48.456609  1.147535  50.056725   
EARL2  Gaussian-EARL2  56.731467  1.257550  54.162311  1.948415  53.180412   
       PseudoEARL-Net  53.401655  0.611866  49.105451  0.838805  52.979102   

                        Lesions             
                            aRE       PSNR  
Target Method                               
EARL1  Gaussian-EARL1  4.184150  30.955267  
       PseudoEARL-Net  1.528528  42.269031  
EARL2  Gaussian-EARL2  1.734601  43.579277  
       PseudoEARL-Net  0.889893  44.863411

In [22]:
final_dict['a100'].__len__()

41

In [ ]:
import pandas as pd
import numpy as np

def generate_delta_suv_table(
    master_dict,
    target_domains=None,
    target_vois=None,
    target_methods=None
):
    """
    Génère un tableau des erreurs de quantification ΔSUV (Max, Mean, Peak).
    
    Paramètres:
    - master_dict : Le dictionnaire extrait (contenant tous les patients).
    - target_domains : Liste des domaines à inclure (ex: ['chb', 'rennes']). Par défaut: Tous.
    - target_vois : Liste des VOIs à inclure. Par défaut: Les 6 classiques.
    - target_methods : Liste des méthodes à inclure. Par défaut: Pseudo et Gaussien.
    """
    
    # 1. Initialisation et configuration des filtres par défaut
    if target_vois is None:
        target_vois = ['Liver', 'Brain', 'Lung', 'Bladder', 'Spleen', 'Lesion']
    if target_methods is None:
        target_methods = ['PseudoEARL-Net', 'Gaussian-EARL']
        
    # Normalisation des domaines fournis par l'utilisateur (ex: 'domain_chb' -> 'chb')
    if target_domains is not None:
        target_domains_clean = [d.lower().replace('domain_', '') for d in target_domains]
    else:
        target_domains_clean = None

    records = []
    
    # 2. Aplatissement du dictionnaire avec application des filtres
    for domain, subjects in master_dict.items():
        domain_clean = domain.lower().replace('domain_', '')
        
        # --- Filtre par domaine ---
        if target_domains_clean is not None and domain_clean not in target_domains_clean:
            continue
            
        for subj, std_data in subjects.items():
            for std_key, voi_data in std_data.items():
                
                for voi, methods_data in voi_data.items():
                    # Normalisation du nom de la VOI
                    if voi.lower() in ['lesion', 'lesions']:
                        display_voi = 'Lesion'
                    elif 'bladder' in voi.lower():
                        display_voi = 'Bladder'
                    else:
                        display_voi = voi.capitalize()
                        
                    # --- Filtre par VOI ---
                    if display_voi not in target_vois:
                        continue
                        
                    # --- Gestion des Lésions (Multi-Composantes) ---
                    if display_voi == 'Lesion':
                        for comp_idx, comp_methods in methods_data.items():
                            for method, metrics in comp_methods.items():
                                if method == 'Standard': continue
                                
                                display_method = "Gaussian-EARL" if "Gaussian" in method else "PseudoEARL-Net"
                                
                                # --- Filtre par méthode ---
                                if display_method not in target_methods:
                                    continue
                                    
                                records.append({
                                    'Domain': domain_clean,
                                    'VOI': display_voi,
                                    'Method': display_method,
                                    'Max_GT': metrics.get('Max_GT', np.nan),
                                    'Max_Pred': metrics.get('Max_Pred', np.nan),
                                    'Mean_GT': metrics.get('Mean_GT', np.nan),
                                    'Mean_Pred': metrics.get('Mean_Pred', np.nan),
                                    'Peak_GT': metrics.get('SUV_Peak_GT', np.nan),
                                    'Peak_Pred': metrics.get('SUV_Peak_Pred', np.nan)
                                })
                                
                    # --- Gestion des autres Organes (Global) ---
                    else:
                        for method, metrics in methods_data.items():
                            if method == 'Standard': continue
                            
                            display_method = "Gaussian-EARL" if "Gaussian" in method else "PseudoEARL-Net"
                            
                            # --- Filtre par méthode ---
                            if display_method not in target_methods:
                                continue
                                
                            records.append({
                                'Domain': domain_clean,
                                'VOI': display_voi,
                                'Method': display_method,
                                'Max_GT': metrics.get('Max_GT', np.nan),
                                'Max_Pred': metrics.get('Max_Pred', np.nan),
                                'Mean_GT': metrics.get('Mean_GT', np.nan),
                                'Mean_Pred': metrics.get('Mean_Pred', np.nan),
                                'Peak_GT': metrics.get('SUV_Peak_GT', np.nan),
                                'Peak_Pred': metrics.get('SUV_Peak_Pred', np.nan)
                            })
                            
    df_flat = pd.DataFrame(records)
    
    if df_flat.empty:
        print("⚠️ Aucune donnée n'a pu être extraite avec ces filtres.")
        return None

    # 3. Calcul des Deltas pour chaque métrique
    for m in ['Max', 'Mean', 'Peak']:
        df_flat[f'Raw_Delta_{m}'] = df_flat[f'{m}_Pred'] - df_flat[f'{m}_GT']
        df_flat[f'Rel_Delta_{m}'] = (df_flat[f'Raw_Delta_{m}'] / df_flat[f'{m}_GT']) * 100
        # df_flat[f'Rel_Delta_{m}'] = np.where(
        #     df_flat[f'{m}_GT'] != 0,
        #     (df_flat[f'Raw_Delta_{m}'] / df_flat[f'{m}_GT']) * 100,
        #     np.nan
        # )

    # 4. Définition de l'ordre d'affichage final (basé sur les requêtes)
    default_vois_order = ['Liver', 'Brain', 'Lung', 'Bladder', 'Spleen', 'Lesion']
    ordered_vois = [v for v in default_vois_order if v in target_vois]
    # Ajout des VOIs exotiques éventuelles à la fin
    for v in target_vois:
        if v not in ordered_vois:
            ordered_vois.append(v)

    default_methods_order = ['PseudoEARL-Net', 'Gaussian-EARL']
    ordered_methods = [m for m in default_methods_order if m in target_methods]
    
    ordered_metrics = ['Max', 'Mean', 'Peak']
    
    table_rows = []
    
    # 5. Agrégation et Formatage
    for voi in ordered_vois:
        row_data = {('VOI', ''): voi}
        
        for method in ordered_methods:
            df_subset = df_flat[(df_flat['VOI'] == voi) & (df_flat['Method'] == method)]
            
            for m in ordered_metrics:
                raw_vals = df_subset[f'Raw_Delta_{m}'].dropna()
                rel_vals = df_subset[f'Rel_Delta_{m}'].dropna()
                
                if len(raw_vals) > 0:
                    raw_mean = raw_vals.mean()
                    raw_std = raw_vals.std(ddof=1) if len(raw_vals) > 1 else 0.0
                    rel_mean = rel_vals.mean()
                    
                    # Format exact: -0.01 ± 0.08 (-0.22%)
                    formatted_str = f"{raw_mean:+.2f} ± {raw_std:.2f} ({rel_mean:+.2f}%)"
                else:
                    formatted_str = "N/A"
                    
                col_key = (method, f"SUV_{m.lower()}")
                row_data[col_key] = formatted_str
                
        table_rows.append(row_data)

    # 6. Création du DataFrame multi-indexé
    multi_columns = pd.MultiIndex.from_tuples(
        [('VOI', '')] + 
        [(method, f"SUV_{m.lower()}") for method in ordered_methods for m in ordered_metrics]
    )
    
    final_df = pd.DataFrame(
        [[row.get(col, '') for col in multi_columns] for row in table_rows],
        columns=multi_columns
    )
    
    final_df = final_df.set_index(('VOI', ''))
    final_df.index.name = None 
    
    return final_df

# =========================================================
# EXEMPLES D'EXÉCUTION
# =========================================================

# table_globale = generate_delta_suv_table(final_dict)
table_globale = generate_delta_suv_table({'a100': final_dict['a100']})
table_globale

PseudoEARL-Net                                                \
                       SUV_max               SUV_mean               SUV_peak   
Liver    +0.04 ± 0.05 (+0.99%)  +0.00 ± 0.00 (+0.05%)  +0.03 ± 0.04 (+0.95%)   
Brain    +0.07 ± 0.07 (+0.60%)  +0.00 ± 0.00 (+0.04%)  +0.07 ± 0.11 (+0.65%)   
Lung     +0.03 ± 0.03 (+0.98%)  +0.00 ± 0.00 (+0.14%)  +0.03 ± 0.09 (+0.76%)   
Bladder  +0.35 ± 0.67 (+0.53%)  +0.04 ± 0.07 (+0.19%)  +0.43 ± 0.77 (+0.91%)   
Spleen   +0.05 ± 0.18 (+0.94%)  +0.00 ± 0.00 (+0.11%)  +0.03 ± 0.05 (+0.77%)   
Lesion   +0.08 ± 0.15 (+1.21%)  +0.03 ± 0.04 (+0.72%)  +0.05 ± 0.09 (+1.09%)   

                 Gaussian-EARL                                                
                       SUV_max               SUV_mean               SUV_peak  
Liver    +0.18 ± 0.18 (+4.22%)  +0.01 ± 0.00 (+0.26%)  +0.08 ± 0.11 (+2.35%)  
Brain    +0.24 ± 0.19 (+2.01%)  +0.01 ± 0.00 (+0.10%)  +0.15 ± 0.19 (+1.38%)  
Lung     +0.09 ± 0.09 (+3.01%)  +0.00 ± 0.00 (+0.47%)  +0.05 ± 0.12 (+1.88%)  
Bladder  +0.83 ± 1.25 (+1.42%)  +0.06 ± 0.05 (+0.33%)  +0.73 ± 0.85 (+1.61%)  
Spleen   +0.13 ± 0.16 (+3.94%)  +0.01 ± 0.01 (+0.48%)  +0.06 ± 0.11 (+1.87%)  
Lesion   +0.17 ± 0.25 (+3.05%)  +0.07 ± 0.07 (+1.94%)  +0.10 ± 0.13 (+2.27%)

In [12]:
[
            "UXKrHrDKshLbyyHd2G5mVp", "TE2swsiFL9BckDDwje5BEm", "Eoozgda4UVszGUcVmMN3qr", "ZBknjLXpYrRkTVD6uoC8Cz", "FqEf7mTqYKrB83vGk6xqQS", "FhjK3fuwuNZ8cb2CQC76DV", "G38MAFQ8QJxmP3RdDgtatM", "2ybL92aNCnoQcHxamqh4r8", "KJxTNtbpCqxZgfKFkLL5bw", "knaDEJdkT7Fyit7Jv83aiX", "2eLoHvwtUjSnYAHKYDWuzw", "HNxD4294tCwor8Q279PadP", "P3CMSYQHbUzsxCqF6naxWq", "nb7SQMC5vKR3wLBjCmobt7", "9MkG9YCPJqjcTNoGMyqZh5", "Y89iNoWEoRWohbsz5B3vWH", "Nrfs8M7unx7fcaSJECYBC3", "CycDYjZVUXgW5ukQUW4TC7", "X8mcdRzrMuzzpfYAMxzozt", "VmYX8qGn8EKjdhvLMsJrs7", "7uK9RmAChXtqziY7pW5Dsa", "Jssbv8wnaAUMY9BAzkzFLr", "aDcCdL2rRSXNSDBc8gbas7", "9MzDqX7RRioqo9rvf9THcD", "SN88PUgr36xHCc9R3HgKNC", "gLBBcHJhNu6YktJxCpRokL", "4V5Yz3ocu6CcwuhVq38LSR", "GQz9nY6LSG2GNcbDnPWWsi", "LH2rQD9ib4KZ5GRSE8AP6q", "W7r3YvDijSJssBRqfPzaCG", "LazJcKFPHizr55EtCj4dDF", "XiVyPLpRQgWLfHY8Wkpw3N", "mw8KJNCixqMhCvxDhrqmt2", "QCUUZfRhj7hnniiVkAXs6B", "2KxS5KpUTLWh3qiQiyPs8J", "FLSCiLeVxA2fwYnzEfctyU", "SYPxsGPFytXdPHmmVxBPDo", "HPdK8qEKMH2jmTk6bGFhTT", "ioPAP7D9tR3C8G4KfVKhA3", "4jzaeDGxuPX4fCfUvDbQuj", "Zj8fVmNfYeAtJnr93L538V", "ZBVNS2t2iZH7wcepE8F4YS", "T3Ma8JKQtBT4LPus4Jxeh3", "KLhQwpbbFaL7t7zLhrHk6p", "YeE6xB28d2U6kn86m2JiBU", "eATSittkopiQUBHr4bMvhV", "WNpLDJcE8kaQt6YXSVcvbw", "4bYLrdGX6Zj569sVydX5uy", "VjghsnARRYj63muf5NYjBv", "ArTMw7c7PQ3zhtpaS4M8fk", "W3G7QPSz9zYCKTi7W6svV5", "gw2kfxahaHTjAK5kzPLmmu", "GpMghx6KCQmcY7c3TS9nEn", "6FUnHSKr2rztBpzNEUvMg4", "3yPTnAs4wZmYDjsY9tvepb", "fMi45AwSLcvi5c7WDud2JG", "Zp6n3YLNQfdVr8s4qPvs5D", "nWhT5ggKSQKZaiahmEiBho", "n5nHWAtDTxGHejhyMvaDgT", "3ePZxLszjdLzWWfvrzxQND", "Piph5SiyBhRTMvGS6WNkKD", "4jJhJSHeiiXLmftGbGHxhS", "ehA9bCaqVajEGu9tPzvnjb", "komWetdTp2nRXqeLvDaAgv", "5s8ZTzvhDkSBZ9vJ3vTAhg"
        ].__len__()

65

In [14]:
138 + 100 + 89 + 65 + 40

432

In [ ]:
# patients sans lung sur a100


In [23]:
' '.join([
            "A100779499", "A100761203", "A100730682", "A100789833", "A100780748", "A100793026", "A100782126", "A100787103", "A100783219", "A100731121", "A100788425", "A100738767", "A100787853", "A100738804", "A100738911", "A100720088", "A100783275", "A100760355", "A100791079", "A100763383", "A100770716", "A100795885", "A100774021", "A100776580", "A100741426", "A100780264", "A100792333", "A100716341", "A100716990", "A100729033", "A100746964", "A100788640", "A100698204", "A100782519", "A100760650", "A100791050", "A100782362", "A100763128", "A100718546", "A100665264", "A100749814", "A100791822", "A100767671", "A100737106", "A100737469", "A100779238", "A100668903", "A100748174", "A100783842", "A100759366", "A100755163", "A100792517", "A100795450", "A100703005", "A100788644", "A100563885", "A100788619", "A100776634", "A100787258", "A100797773", "A100756834", "A100782522", "A100563684", "A100739006", "A100678049", "A100750063", "A100740690", "A100739271", "A100720493", "A100787829", "A100721721", "A100713233", "A100792727", "A100784916", "A100730672", "A100734679", "A100789525", "A100786185", "A100792537", "A100718770", "A100770019", "A100794227", "A100709777", "A100686969", "A100788105", "A100790091", "A100706969", "A100764406", "A100710419", "A100784481", "A100787949", "A100737144", "A100752914", "A100748980", "A100789635", "A100670828", "A100793815", "A100728424", "A100782941", "A100791891", "A100736488", "A100759377", "A100672908", "A100791867", "A100765170", "A100785598", "A100551319", "A100794741", "A100786953", "V00788328", "A100767479", "A100680483", "A100711849", "A100788658", "A100765061", "A100738988", "A100733608", "A100757158", "A100791856", "A100788053", "A100787556", "A100775880", "A100743451", "A100793684", "A100666066", "A100787400", "A100758718", "A100794788", "A100792034", "A100737608", "A100792741", "A100765772", "A100778077", "A100759384", "A100763890", "A100772705", "A100681810", "A100782198"
        ])

'A100779499 A100761203 A100730682 A100789833 A100780748 A100793026 A100782126 A100787103 A100783219 A100731121 A100788425 A100738767 A100787853 A100738804 A100738911 A100720088 A100783275 A100760355 A100791079 A100763383 A100770716 A100795885 A100774021 A100776580 A100741426 A100780264 A100792333 A100716341 A100716990 A100729033 A100746964 A100788640 A100698204 A100782519 A100760650 A100791050 A100782362 A100763128 A100718546 A100665264 A100749814 A100791822 A100767671 A100737106 A100737469 A100779238 A100668903 A100748174 A100783842 A100759366 A100755163 A100792517 A100795450 A100703005 A100788644 A100563885 A100788619 A100776634 A100787258 A100797773 A100756834 A100782522 A100563684 A100739006 A100678049 A100750063 A100740690 A100739271 A100720493 A100787829 A100721721 A100713233 A100792727 A100784916 A100730672 A100734679 A100789525 A100786185 A100792537 A100718770 A100770019 A100794227 A100709777 A100686969 A100788105 A100790091 A100706969 A100764406 A100710419 A100784481 A10078794